In [1]:
import numpy as np
from scipy.special import jv
from scipy.optimize import brentq
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh
from scipy.interpolate import CubicSpline
from joblib import Parallel, delayed, parallel_backend

def compute_G_vs_t_for_kappas(
    kappas,           # lista o array de κ en nm/s
    t_values,         # array de tiempos (s)
    d, D0, D_s, R, L, delta, lambda_param,
    alpha_grid,       # parámetro de estiramiento
    n_modes_radial, m_max,
    N_quad           # número de nodos de cuadratura (ρ, z)
):
    results = {}

    # 1) Malla axial adaptativa para la parte de Sturm–Liouville
    N_z = N_quad
    j_idx = np.arange(N_z)
    z_ax = d + L * np.sinh(alpha_grid * j_idx / (N_z - 1)) / np.sinh(alpha_grid)
    h = np.diff(z_ax)  # tamaño N_z-1

    # 2) Definir D(z) como en tu cuaderno
    z_c = lambda_param * L
    def D_of_z(z_vals):
        S = 0.5 * (1 + np.tanh((z_vals - (z_c + d)) / delta))
        return D_s + (D0 - D_s) * ((1 - np.exp(-lambda_param * (z_vals - d))) * (1 - S) + S)

    # 3) Definir función trascendental radial y D_eff
    D_vals_ax = D_of_z(z_ax)
    D_eff = L / np.trapz(1.0 / D_vals_ax, z_ax)

    def robin_eq(alpha, kappa):
        return alpha * jv(1, alpha * R) - (kappa / D_eff) * jv(0, alpha * R)

    # 4) Función que, dado un α_n y κ, arma y resuelve el problema axial:
    def solve_axial_modes(alpha_n, kappa):
        # Construir diag y off para cada modo n
        eigenvalues_ax = np.zeros((n_modes_radial, m_max))
        eigenvectors_ax = np.zeros((n_modes_radial, m_max, N_z))
        for n in range(n_modes_radial):
            diag = np.zeros(N_z)
            off  = np.zeros(N_z - 1)
            D_j   = D_vals_ax  # D(z_j) directamente
            D_jp2 = 2 * D_j[:-1] * D_j[1:] / (D_j[:-1] + D_j[1:])  # D_{j+1/2}

            # Interiores j=1..N_z-2
            for j in range(1, N_z - 1):
                Djm = D_jp2[j - 1]
                Djp = D_jp2[j]
                hj  = h[j]
                hjm = h[j - 1]
                hjp = h[j]
                diag[j] = (Djm / (hj * hjm) +
                           Djp / (hj * hjp) +
                           alpha_n[n]**2 * D_j[j])
                off[j] = -Djp / (hj * hjp)

            # Contorno j=0 (Robin superficie)
            D0_val = D_j[0]
            h0     = h[0]
            diag[0] = 2 * D0_val / h0**2 + 2 * kappa / h0 + alpha_n[n]**2 * D0_val
            off[0]  = -2 * D0_val / h0**2

            # Contorno j=N_z-1 (Robin tope)
            DL_val = D_j[-1]
            hL     = h[-1]
            diag[-1]  = DL_val / hL**2 + kappa / hL + alpha_n[n]**2 * DL_val
            off[-2]   = -DL_val / hL**2

            # Montar matriz esparsa tridiagonal
            A_n = diags(
                diagonals=[off, diag, off],
                offsets=[-1, 0, 1],
                shape=(N_z, N_z),
                format="csr"
            )

            # Resolver valores propios axiales (v0=None para reiniciar cada vez)
            vals, vecs = eigsh(A_n, k=m_max, sigma=0.0, which='LM', tol=1e-8, v0=None)
            idx_sort = np.argsort(vals)
            eigenvalues_ax[n, :]    = vals[idx_sort]
            eigenvectors_ax[n, :, :] = vecs[:, idx_sort].T

        return eigenvalues_ax, eigenvectors_ax

    # ------------------------------------------------------------
    # Loop principal sobre cada valor de kappa:
    for kappa in kappas:
        # 5) Encontrar raíces α_n para este kappa
        alpha_roots = []
        alpha_cur = 0.0
        step = 1e-4
        while len(alpha_roots) < n_modes_radial:
            try:
                root = brentq(robin_eq, alpha_cur, alpha_cur + step, args=(kappa,), xtol=1e-6)
                alpha_roots.append(root)
                alpha_cur = root + step
            except ValueError:
                alpha_cur += step
        alpha_n = np.array(alpha_roots)

        # 6) Resolver modos axiales en paralelo
        with parallel_backend('loky'):
            eigenvalues_ax, eigenvectors_ax = solve_axial_modes(alpha_n, kappa)

        # 7) Normalizar modos axiales en z
        norms = np.sqrt(np.trapz(eigenvectors_ax.astype(np.float32)**2, z_ax, axis=2))
        eigenvectors_ax /= norms[:, :, None]

        # 8) Construir coeficientes A_nm
        rho0 = R / 2.0
        z0   = (z_c+d)/2
        R0_mat, _ = R_func(alpha_n, np.array([rho0]))  # shape (n_modes_radial, 1)
        R0 = R0_mat[:, 0]
        Z_interps = np.empty((n_modes_radial, m_max), dtype=object)
        for n in range(n_modes_radial):
            for m in range(m_max):
                Z_interps[n, m] = CubicSpline(z_ax, eigenvectors_ax[n, m, :], extrapolate=False)

        Z0 = np.zeros((n_modes_radial, m_max))
        for n in range(n_modes_radial):
            for m in range(m_max):
                Z0[n, m] = Z_interps[n, m](z0)
        A_nm = R0[:, None] * Z0  # shape (n_modes_radial, m_max)

        # 9) Preparar cuadratura para G(t)
        rho_nodes = np.linspace(0.0, R, N_quad)
        z_nodes   = np.linspace(d, L + d, N_quad)
        dr = rho_nodes[1] - rho_nodes[0]
        dz = z_nodes[1]   - z_nodes[0]
        J0_mat  = jv(0, alpha_n[:, None] * rho_nodes[None, :])  # (n_modes_radial, N_quad)

        Z_tens = np.empty((N_quad, n_modes_radial, m_max))
        for n in range(n_modes_radial):
            for m in range(m_max):
                Z_tens[:, n, m] = Z_interps[n, m](z_nodes)

        R2 = rho_nodes**2
        Z2 = (z_nodes + 1e-30)**2
        f_mat = (3 * Z2[None, :] - R2[:, None]) / (R2[:, None] + Z2[None, :])**2.5

        # 10) Calcular G(t) para todos los t_values
        # T_decay: (N_t, n_modes_radial, m_max)
        T_decay = np.exp(-np.abs(eigenvalues_ax)[None, :, :] * t_values[:, None, None])
        W = T_decay * A_nm[None, :, :]                          # (N_t, n_modes_radial, m_max)
        B = np.einsum('znm,tnm->tzn', Z_tens, W)                 # (N_t, N_quad, n_modes_radial)
        S = np.einsum('nr,tzn->trz', J0_mat, B)                  # (N_t, N_quad, N_quad)

        integrand = rho_nodes[None, :, None] * f_mat[None, :, :] * S  # (N_t, N_quad, N_quad)
        ints = integrand.sum(axis=(1, 2)) * dr * dz                    # (N_t,)
        G_vals = (5.0 / (16.0 * np.pi)) * ints                         # (N_t,)

        # Guardar en el diccionario
        results[kappa] = G_vals

    return results

In [2]:
def R_func(alpha, rho):
    """
    Calcula R_n(rho) = J0(alpha_n * rho) normalizados con la norma analítica
      ∫₀ᴿ J₀²(αₙ ρ) ρ dρ = (R²/2)[J₀²(αₙR) + J₁²(αₙR)].

    Args:
        alpha_n: array de raíces αₙ, shape (n_modes,)
        rho_vals: escalar o array de posiciones radiales, shape (m,)

    Returns:
        R_normed: array (n_modes, m) de los modos normalizados
        norms:    array (n_modes,) con el valor exacto de la norma
    """
   # 1) Aseguramos que alpha_n y rho_vals sean arrays 1D
    alpha_arr = np.atleast_1d(alpha)              # shape (n_modes,)
    rho_arr   = np.atleast_1d(rho)             # shape (m,)

    # 2) Creamos vistas 2D para broadcasting
    alpha = alpha_arr.reshape(-1, 1)                # (n_modes, 1)
    rho   = rho_arr.reshape(1, -1)                  # (1, m)

    # 3) Evaluamos el modo sin normalizar
    R_matrix = jv(0, alpha * rho)                   # (n_modes, m)

    # 4) Calculamos la norma analítica usando alpha_arr (no el escalar viejo)
    J0R = jv(0, alpha_arr * R)                      # (n_modes,)
    J1R = jv(1, alpha_arr * R)                    # (n_modes,)
    norms = 0.5 * R**2 * (J0R**2 + J1R**2)           # (n_modes,)

    # 5) Aseguramos que norms sea un array 1D
    norms = np.atleast_1d(norms)

    # 6) Factor de normalización
    N = 1.0 / np.sqrt(norms)                        # (n_modes,)

    # 7) Aplicamos normalización por filas
    R_normed = N[:, None] * R_matrix                # (n_modes, m)

    return R_normed, np.sqrt(norms)

In [3]:
# 1) Definir todos los parámetros físicos (igual que en tu cuaderno):
d             = 15.0
D0            = 2e6
D_s           = 2e4
R             = 100 * d
L             = 100 * d
delta         = 2 * d / 3
lambda_param  = 0.75 / d
alpha_grid    = 5.0
n_modes_radial = 50
m_max          = 50
N_quad         = 1000

# 2) Vector de tiempos donde evaluar G(t):
t_values = np.logspace(-7, -1, N_quad)

# 3) Lista de kappa que quieres barrer:
kappas_list = [0.0, 15.0, 30.0, 50.0, 100.0,200.0,500.0,1000.0,2000.0,5000.0]

# 4) Llamar a la función:
results_dict = compute_G_vs_t_for_kappas(
    kappas=kappas_list,
    t_values=t_values,
    d=d, D0=D0, D_s=D_s, R=R, L=L,
    delta=delta, lambda_param=lambda_param,
    alpha_grid=alpha_grid,
    n_modes_radial=n_modes_radial,
    m_max=m_max,
    N_quad=N_quad
)

# Cada clave en results_dict es un valor de kappa, 
# y results_dict[kappa] es el array G(t) de longitud len(t_values).

# 5) Guardar en disco en formato de texto (si quieres):
import numpy as np

header = "Tiempo(s)"
data_to_write = t_values.reshape(-1, 1)
for kappa in kappas_list:
    header += f"   G(t)_kappa{kappa}"
    data_to_write = np.hstack([data_to_write, results_dict[kappa].reshape(-1, 1)])

np.savetxt("data/VariateKappa_G_vs_t.dat", data_to_write, header=header, fmt="%.9e")